In [8]:
import time
import json
import csv
import re
import os
import threading
from datetime import datetime
from urllib.parse import quote, urlparse
from calendar import monthrange
import tkinter as tk
from tkinter import ttk, scrolledtext, messagebox, filedialog

# 선택적 import
try:
    from selenium import webdriver
    from selenium.webdriver.common.by import By
    from selenium.webdriver.chrome.service import Service
    from selenium.webdriver.chrome.options import Options
    SELENIUM_AVAILABLE = True
except ImportError:
    SELENIUM_AVAILABLE = False

try:
    import pandas as pd
    from openpyxl import Workbook, load_workbook
    EXCEL_AVAILABLE = True
except ImportError:
    EXCEL_AVAILABLE = False

class MenuImageCollectorGUI:
    def __init__(self, root):
        self.root = root
        self.root.title("음식 이미지 수집기")
        self.root.geometry("600x700")
        
        self.driver = None
        self.is_running = False
        self.log_file = None
        
        self.setup_gui()
        self.check_csv_file()
    
    def setup_gui(self):
        """GUI 구성"""
        # 메인 프레임
        main_frame = ttk.Frame(self.root, padding="10")
        main_frame.grid(row=0, column=0, sticky=(tk.W, tk.E, tk.N, tk.S))
        
        # 제목
        title_label = ttk.Label(main_frame, text="음식 이미지 수집기", font=("Arial", 16, "bold"))
        title_label.grid(row=0, column=0, columnspan=3, pady=(0, 20))
        
        # CSV 파일 정보
        ttk.Label(main_frame, text="CSV 파일:", font=("Arial", 10, "bold")).grid(row=1, column=0, sticky=tk.W, pady=5)
        self.csv_info_label = ttk.Label(main_frame, text="확인 중...", foreground="gray")
        self.csv_info_label.grid(row=1, column=1, columnspan=2, sticky=tk.W, pady=5)
        
        # 구분선
        separator1 = ttk.Separator(main_frame, orient='horizontal')
        separator1.grid(row=2, column=0, columnspan=3, sticky=(tk.W, tk.E), pady=10)
        
        # 기간 설정
        period_frame = ttk.LabelFrame(main_frame, text="수집 기간 설정", padding="10")
        period_frame.grid(row=3, column=0, columnspan=3, sticky=(tk.W, tk.E), pady=10)
        
        # 시작 날짜
        ttk.Label(period_frame, text="시작:").grid(row=0, column=0, sticky=tk.W)
        self.start_year = tk.StringVar(value="2024")
        self.start_month = tk.StringVar(value="6")
        
        ttk.Entry(period_frame, textvariable=self.start_year, width=6).grid(row=0, column=1, padx=5)
        ttk.Label(period_frame, text="년").grid(row=0, column=2)
        ttk.Entry(period_frame, textvariable=self.start_month, width=4).grid(row=0, column=3, padx=5)
        ttk.Label(period_frame, text="월").grid(row=0, column=4)
        
        # 끝 날짜
        ttk.Label(period_frame, text="끝:").grid(row=1, column=0, sticky=tk.W, pady=(10, 0))
        self.end_year = tk.StringVar(value="2024")
        self.end_month = tk.StringVar(value="6")
        
        ttk.Entry(period_frame, textvariable=self.end_year, width=6).grid(row=1, column=1, padx=5, pady=(10, 0))
        ttk.Label(period_frame, text="년").grid(row=1, column=2, pady=(10, 0))
        ttk.Entry(period_frame, textvariable=self.end_month, width=4).grid(row=1, column=3, padx=5, pady=(10, 0))
        ttk.Label(period_frame, text="월").grid(row=1, column=4, pady=(10, 0))
        
        # 구분선
        separator2 = ttk.Separator(main_frame, orient='horizontal')
        separator2.grid(row=4, column=0, columnspan=3, sticky=(tk.W, tk.E), pady=10)
        
        # 옵션 설정
        options_frame = ttk.LabelFrame(main_frame, text="설정", padding="10")
        options_frame.grid(row=5, column=0, columnspan=3, sticky=(tk.W, tk.E), pady=10)
        
        # 저장 위치
        ttk.Label(options_frame, text="저장 위치:").grid(row=0, column=0, sticky=tk.W)
        self.save_path = tk.StringVar(value=os.getcwd())
        ttk.Entry(options_frame, textvariable=self.save_path, width=40).grid(row=0, column=1, padx=5)
        ttk.Button(options_frame, text="찾기", command=self.browse_folder).grid(row=0, column=2)
        
        # 테스트 모드
        self.test_mode = tk.BooleanVar(value=False)
        ttk.Checkbutton(options_frame, text="테스트 모드 (처음 10개 메뉴만)", 
                       variable=self.test_mode).grid(row=1, column=0, columnspan=3, sticky=tk.W, pady=5)
        
        # 구분선
        separator3 = ttk.Separator(main_frame, orient='horizontal')
        separator3.grid(row=6, column=0, columnspan=3, sticky=(tk.W, tk.E), pady=10)
        
        # 버튼
        button_frame = ttk.Frame(main_frame)
        button_frame.grid(row=7, column=0, columnspan=3, pady=10)
        
        self.start_button = ttk.Button(button_frame, text="수집 시작", command=self.start_collection)
        self.start_button.pack(side=tk.LEFT, padx=5)
        
        self.stop_button = ttk.Button(button_frame, text="중지", command=self.stop_collection, state=tk.DISABLED)
        self.stop_button.pack(side=tk.LEFT, padx=5)
        
        ttk.Button(button_frame, text="종료", command=self.root.quit).pack(side=tk.LEFT, padx=5)
        
        # 진행률
        self.progress_var = tk.StringVar(value="대기 중")
        ttk.Label(main_frame, textvariable=self.progress_var).grid(row=8, column=0, columnspan=3, pady=5)
        
        self.progress_bar = ttk.Progressbar(main_frame, mode='determinate')
        self.progress_bar.grid(row=9, column=0, columnspan=3, sticky=(tk.W, tk.E), pady=5)
        
        # 로그 창
        log_frame = ttk.LabelFrame(main_frame, text="진행 로그", padding="5")
        log_frame.grid(row=10, column=0, columnspan=3, sticky=(tk.W, tk.E, tk.N, tk.S), pady=10)
        
        self.log_text = scrolledtext.ScrolledText(log_frame, height=15, state=tk.DISABLED)
        self.log_text.pack(fill=tk.BOTH, expand=True)
        
        # 그리드 가중치 설정
        self.root.columnconfigure(0, weight=1)
        self.root.rowconfigure(0, weight=1)
        main_frame.columnconfigure(1, weight=1)
        main_frame.rowconfigure(10, weight=1)
        
    def browse_folder(self):
        """폴더 선택"""
        folder = filedialog.askdirectory(initialdir=self.save_path.get())
        if folder:
            self.save_path.set(folder)
    
    def check_csv_file(self):
        """CSV 파일 확인"""
        csv_file = "식당대12중53소132상세메뉴379분류.csv"
        if os.path.exists(csv_file):
            try:
                menu_count = self.count_menus(csv_file)
                self.csv_info_label.config(text=f"{csv_file} ({menu_count}개 메뉴)", foreground="green")
            except Exception as e:
                self.csv_info_label.config(text=f"{csv_file} (읽기 오류)", foreground="red")
        else:
            self.csv_info_label.config(text="파일 없음", foreground="red")
    
    def count_menus(self, csv_file):
        """메뉴 개수 카운트"""
        with open(csv_file, 'r', encoding='utf-8') as file:
            csv_reader = csv.DictReader(file)
            menus = set()
            for row in csv_reader:
                if '상세메뉴' in row and row['상세메뉴']:
                    items = [item.strip() for item in row['상세메뉴'].split(',') if item.strip()]
                    menus.update(items)
            return len(menus)
    
    def log(self, message):
        """로그 출력"""
        timestamp = datetime.now().strftime("%H:%M:%S")
        log_message = f"[{timestamp}] {message}"
        
        # GUI 로그창에 출력
        self.log_text.config(state=tk.NORMAL)
        self.log_text.insert(tk.END, log_message + "\n")
        self.log_text.see(tk.END)
        self.log_text.config(state=tk.DISABLED)
        
        # 파일에 로그 저장
        if self.log_file:
            try:
                with open(self.log_file, 'a', encoding='utf-8') as f:
                    f.write(log_message + "\n")
            except:
                pass
        
        # GUI 업데이트
        self.root.update_idletasks()
    
    def validate_input(self):
        """입력 유효성 검사"""
        try:
            start_year = int(self.start_year.get())
            start_month = int(self.start_month.get())
            end_year = int(self.end_year.get())
            end_month = int(self.end_month.get())
            
            if not (1 <= start_month <= 12 and 1 <= end_month <= 12):
                messagebox.showerror("입력 오류", "월은 1-12 사이여야 합니다.")
                return None
            
            start_date = datetime(start_year, start_month, 1)
            end_date = datetime(end_year, end_month, 1)
            
            if start_date > end_date:
                messagebox.showerror("입력 오류", "시작 날짜가 끝 날짜보다 늦습니다.")
                return None
            
            return {
                'start_year': start_year,
                'start_month': start_month,
                'end_year': end_year,
                'end_month': end_month
            }
            
        except ValueError:
            messagebox.showerror("입력 오류", "연도와 월은 숫자여야 합니다.")
            return None
    
    def start_collection(self):
        """수집 시작"""
        if not SELENIUM_AVAILABLE:
            messagebox.showerror("오류", "Selenium이 설치되지 않았습니다.\npip install selenium webdriver-manager")
            return
        
        # 입력 검증
        config = self.validate_input()
        if not config:
            return
        
        # CSV 파일 확인
        csv_file = "식당대12중53소132상세메뉴379분류.csv"
        if not os.path.exists(csv_file):
            messagebox.showerror("오류", f"CSV 파일을 찾을 수 없습니다:\n{csv_file}")
            return
        
        # 확인 대화상자
        months = self.generate_months(config['start_year'], config['start_month'], 
                                    config['end_year'], config['end_month'])
        test_text = " (테스트 모드)" if self.test_mode.get() else ""
        
        message = f"수집을 시작하시겠습니까?\n\n"
        message += f"기간: {config['start_year']}-{config['start_month']:02d} ~ {config['end_year']}-{config['end_month']:02d}\n"
        message += f"처리 월수: {len(months)}개월{test_text}\n"
        message += f"저장 위치: {self.save_path.get()}"
        
        if not messagebox.askyesno("확인", message):
            return
        
        # 상태 변경
        self.is_running = True
        self.start_button.config(state=tk.DISABLED)
        self.stop_button.config(state=tk.NORMAL)
        
        # 로그 파일 설정
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        self.log_file = os.path.join(self.save_path.get(), f"collection_log_{timestamp}.txt")
        
        # 별도 스레드에서 실행
        self.collection_thread = threading.Thread(target=self.run_collection, args=(config, csv_file))
        self.collection_thread.daemon = True
        self.collection_thread.start()
    
    def run_collection(self, config, csv_file):
        """수집 실행 (별도 스레드)"""
        try:
            self.log("음식 이미지 수집 시작")
            
            # 드라이버 설정
            if not self.setup_driver():
                return
            
            # 메뉴 로드
            menus = self.load_menus(csv_file)
            if not menus:
                self.log("메뉴 로드 실패")
                return
            
            # 테스트 모드
            if self.test_mode.get():
                menus = menus[:10]
                self.log(f"테스트 모드: {len(menus)}개 메뉴만 처리")
            
            # 월별 범위
            months = self.generate_months(config['start_year'], config['start_month'],
                                        config['end_year'], config['end_month'])
            
            self.log(f"처리 대상: {len(months)}개월, {len(menus)}개 메뉴")
            
            # 진행률 설정
            total_tasks = len(months) * len(menus)
            current_task = 0
            
            # 월별 처리
            saved_files = []
            
            for month in months:
                if not self.is_running:
                    break
                
                self.log(f"{month['display']} 처리 시작")
                self.progress_var.set(f"{month['display']} 처리 중...")
                
                results = []
                month_start_time = time.time()
                
                for i, menu in enumerate(menus, 1):
                    if not self.is_running:
                        break
                    
                    current_task += 1
                    progress = (current_task / total_tasks) * 100
                    self.progress_bar['value'] = progress
                    
                    # 메뉴 처리
                    result = self.collect_menu_images(menu, month['start_date'], month['end_date'])
                    results.append(result)
                    
                    # 진행률 업데이트
                    if i % 10 == 0 or i <= 5:
                        self.progress_var.set(f"{month['display']} - {i}/{len(menus)} ({progress:.1f}%)")
                    
                    time.sleep(1)
                
                if self.is_running:
                    # 월별 결과 저장
                    filename = self.save_results(results, month['display'])
                    if filename:
                        saved_files.append(filename)
                    
                    # 월별 요약
                    month_time = time.time() - month_start_time
                    total_search = sum(r.get('total_count', 0) for r in results)
                    total_urls = sum(len(r.get('urls', [])) for r in results)
                    
                    self.log(f"{month['display']} 완료 - {month_time/60:.1f}분")
                    self.log(f"  검색: {total_search:,}개, 수집: {total_urls:,}개")
            
            if self.is_running:
                self.log(f"전체 작업 완료! 생성된 파일: {len(saved_files)}개")
                self.progress_var.set("완료")
                self.progress_bar['value'] = 100
                messagebox.showinfo("완료", f"수집이 완료되었습니다!\n생성된 파일: {len(saved_files)}개")
            else:
                self.log("작업이 중단되었습니다.")
                self.progress_var.set("중단됨")
                
        except Exception as e:
            self.log(f"오류 발생: {e}")
            messagebox.showerror("오류", f"수집 중 오류가 발생했습니다:\n{e}")
        finally:
            self.cleanup()
    
    def setup_driver(self):
        """드라이버 설정"""
        try:
            try:
                from webdriver_manager.chrome import ChromeDriverManager
                service = Service(ChromeDriverManager().install())
            except ImportError:
                service = Service()
            
            chrome_options = Options()
            chrome_options.add_argument('--headless')
            chrome_options.add_argument('--no-sandbox')
            chrome_options.add_argument('--disable-dev-shm-usage')
            chrome_options.add_argument('--disable-gpu')
            chrome_options.add_argument('--window-size=1920,1080')
            
            self.driver = webdriver.Chrome(service=service, options=chrome_options)
            self.log("Chrome 드라이버 설정 완료")
            return True
            
        except Exception as e:
            self.log(f"드라이버 설정 실패: {e}")
            messagebox.showerror("오류", f"Chrome 드라이버 설정에 실패했습니다:\n{e}")
            return False
    
    def load_menus(self, csv_file):
        """메뉴 로드"""
        try:
            with open(csv_file, 'r', encoding='utf-8') as file:
                csv_reader = csv.DictReader(file)
                menus = []
                seen = set()
                
                for row in csv_reader:
                    if '상세메뉴' in row and row['상세메뉴']:
                        items = [item.strip() for item in row['상세메뉴'].split(',') if item.strip()]
                        for item in items:
                            if item not in seen and len(item) >= 2:
                                seen.add(item)
                                menus.append(item)
                
                self.log(f"메뉴 로드 완료: {len(menus)}개")
                return menus
                
        except Exception as e:
            self.log(f"메뉴 로드 실패: {e}")
            return []
    
    def generate_months(self, start_year, start_month, end_year, end_month):
        """월별 범위 생성"""
        months = []
        current = datetime(start_year, start_month, 1)
        end = datetime(end_year, end_month, 1)
        
        while current <= end:
            year = current.year
            month = current.month
            
            first_day = datetime(year, month, 1)
            last_day = datetime(year, month, monthrange(year, month)[1])
            
            months.append({
                'year': year,
                'month': month,
                'start_date': first_day.strftime('%Y%m%d'),
                'end_date': last_day.strftime('%Y%m%d'),
                'display': f"{year}-{month:02d}"
            })
            
            if month == 12:
                current = datetime(year + 1, 1, 1)
            else:
                current = datetime(year, month + 1, 1)
        
        return months
    
    def collect_menu_images(self, menu, start_date, end_date):
        """메뉴별 이미지 수집"""
        if not self.driver or not self.is_running:
            return {'menu': menu, 'total_count': 0, 'urls': []}
        
        try:
            query = f"음식 {menu}"
            
            # 검색 URL
            nso_param = f"so:r,p:from{start_date}to{end_date}"
            url = f"https://search.naver.com/search.naver?where=image&query={quote(query)}&sm=tab_opt&nso={quote(nso_param)}&mode=normal&section=image&ccl=0&gif=0"
            
            self.driver.get(url)
            time.sleep(2)
            
            # 검색 결과 개수
            total_count = self.get_search_count()
            
            if total_count == 0:
                return {'menu': menu, 'total_count': 0, 'urls': []}
            
            # URL 수집
            urls = self.collect_urls()
            
            return {
                'menu': menu,
                'total_count': total_count,
                'urls': urls
            }
            
        except Exception as e:
            return {'menu': menu, 'total_count': 0, 'urls': [], 'error': str(e)}
    
    def get_search_count(self):
        """검색 결과 개수 추출"""
        try:
            text = self.driver.find_element(By.TAG_NAME, 'body').text
            matches = re.findall(r'약\s*([\d,]+)\s*개', text)
            if matches:
                return int(matches[0].replace(',', ''))
            return 0
        except:
            return 0
    
    def collect_urls(self):
        """URL 수집"""
        urls = set()
        
        for _ in range(10):  # 최대 10번 스크롤
            if not self.is_running:
                break
            
            try:
                elements = self.driver.find_elements(By.CSS_SELECTOR, 'img[src*="http"], [data-src*="http"]')
                for elem in elements:
                    for attr in ['src', 'data-src']:
                        url = elem.get_attribute(attr)
                        if url and self.is_valid_url(url):
                            urls.add(url)
                
                self.driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
                time.sleep(1)
                
            except:
                break
        
        return list(urls)
    
    def is_valid_url(self, url):
        """유효한 URL 확인"""
        if not url or len(url) < 20:
            return False
        
        url_lower = url.lower()
        
        if any(word in url_lower for word in ['icon', 'logo', 'banner', 'ad']):
            return False
        
        return any(word in url_lower for word in ['blogfiles', 'postfiles', 'pstatic', 'jpg', 'png', 'webp'])
    
    def save_results(self, results, month_display):
        """결과 저장"""
        timestamp = datetime.now().strftime("%Y%m%d_%H%M")
        filename = f"menu_images_{month_display.replace('-', '')}_{timestamp}"
        
        try:
            if EXCEL_AVAILABLE:
                excel_file = os.path.join(self.save_path.get(), filename + ".xlsx")
                
                # 요약 데이터
                summary = []
                url_list = []
                
                for result in results:
                    menu = result.get('menu', '')
                    total = result.get('total_count', 0)
                    urls = result.get('urls', [])
                    
                    summary.append({
                        '메뉴': menu,
                        '검색결과': total,
                        'URL수': len(urls),
                        '수집률': f"{len(urls)/total*100:.1f}%" if total > 0 else "0%"
                    })
                    
                    for i, url in enumerate(urls, 1):
                        url_list.append({
                            '메뉴': menu,
                            '번호': i,
                            'URL': url,
                            '도메인': urlparse(url).netloc
                        })
                
                # 엑셀 저장
                with pd.ExcelWriter(excel_file, engine='openpyxl') as writer:
                    pd.DataFrame(summary).to_excel(writer, sheet_name='요약', index=False)
                    if url_list:
                        pd.DataFrame(url_list).to_excel(writer, sheet_name='URL목록', index=False)
                
                self.log(f"저장: {os.path.basename(excel_file)}")
                return excel_file
            
        except Exception as e:
            self.log(f"저장 오류: {e}")
        
        # JSON으로 대체 저장
        try:
            json_file = os.path.join(self.save_path.get(), filename + ".json")
            with open(json_file, 'w', encoding='utf-8') as f:
                json.dump(results, f, ensure_ascii=False, indent=2)
            self.log(f"저장: {os.path.basename(json_file)}")
            return json_file
        except Exception as e:
            self.log(f"JSON 저장 실패: {e}")
            return None
    
    def stop_collection(self):
        """수집 중지"""
        self.is_running = False
        self.stop_button.config(state=tk.DISABLED)
        self.log("수집 중지 요청됨...")
    
    def cleanup(self):
        """정리"""
        self.is_running = False
        
        if self.driver:
            try:
                self.driver.quit()
                self.log("브라우저 종료")
            except:
                pass
            self.driver = None
        
        # 버튼 상태 복원
        self.start_button.config(state=tk.NORMAL)
        self.stop_button.config(state=tk.DISABLED)

def main():
    # tkinter 애플리케이션 시작
    root = tk.Tk()
    app = MenuImageCollectorGUI(root)
    
    # 종료 시 정리
    def on_closing():
        if app.is_running:
            if messagebox.askokcancel("종료", "수집이 진행 중입니다. 종료하시겠습니까?"):
                app.cleanup()
                root.destroy()
        else:
            app.cleanup()
            root.destroy()
    
    root.protocol("WM_DELETE_WINDOW", on_closing)
    root.mainloop()

if __name__ == "__main__":
    print("음식 이미지 수집기 GUI")
    print("필요 패키지: pip install selenium webdriver-manager pandas openpyxl")
    main()

음식 이미지 수집기 GUI
필요 패키지: pip install selenium webdriver-manager pandas openpyxl
